1BM24CS119


---

Hriday R Jain

8 Puzzle using A* search algorithm using misplaced tiles


In [5]:
import heapq

# Define the custom Goal Board globally
GOAL_BOARD = (1, 2, 3, 8, 0, 4, 7, 6, 5)

class PuzzleState:
    def __init__(self, board, g, parent=None, move=""):
        self.board = board       # 1D tuple representing 3x3 grid
        self.g = g               # Cost from start to current node
        self.parent = parent     # Pointer to parent state
        self.move = move         # Move that led to this state
        self.h = self.calculate_misplaced()
        self.f = self.g + self.h

    def calculate_misplaced(self):
        """Calculates Misplaced Tiles heuristic."""
        count = 0
        for i in range(9):
            # Only count numbered tiles (exclude the blank space 0)
            if self.board[i] != 0 and self.board[i] != GOAL_BOARD[i]:
                count += 1
        return count

    def get_neighbors(self):
        """Generates all valid successor states."""
        neighbors = []
        zero_idx = self.board.index(0)
        r, c = zero_idx // 3, zero_idx % 3

        # Directions: (row_offset, col_offset, action_name)
        moves = [(-1, 0, "Up"), (1, 0, "Down"), (0, -1, "Left"), (0, 1, "Right")]

        for dr, dc, move_name in moves:
            nr, nc = r + dr, c + dc
            if 0 <= nr < 3 and 0 <= nc < 3:
                neighbor_idx = nr * 3 + nc
                # Swap elements to create a new board
                new_board = list(self.board)
                new_board[zero_idx], new_board[neighbor_idx] = new_board[neighbor_idx], new_board[zero_idx]
                neighbors.append(PuzzleState(tuple(new_board), self.g + 1, self, move_name))
        return neighbors

    def __lt__(self, other):
        # Tie-breaker for the priority queue based on total cost f(n)
        return self.f < other.f

def solve_8_puzzle(start_board):
    # Priority queue (Open List) and Visited set (Closed List)
    open_list = []
    visited = set()

    start_state = PuzzleState(start_board, 0)
    heapq.heappush(open_list, start_state)

    while open_list:
        current_state = heapq.heappop(open_list)

        if current_state.board == GOAL_BOARD:
            # Reconstruct path along with tracking evaluation metrics
            path = []
            curr = current_state
            while curr:
                path.append((curr.move, curr.board, curr.f, curr.g, curr.h))
                curr = curr.parent
            return path[::-1] # Return from start state to goal state

        visited.add(current_state.board)

        for neighbor in current_state.get_neighbors():
            if neighbor.board not in visited:
                heapq.heappush(open_list, neighbor)

    return None

def print_board(board):
    for i in range(0, 9, 3):
        print(f"{board[i]} {board[i+1]} {board[i+2]}")

# --- Example Execution ---
initial_layout = (2, 8, 3, 1, 6, 4, 7, 0, 5)

solution_steps = solve_8_puzzle(initial_layout)

if solution_steps:
    # Safely unpack the initial layout details from index 0
    start_step = solution_steps[0]
    print("Initial Board State:")
    print_board(start_step[1])
    print(f"Metrics -> f(n): {start_step[2]}, g(n): {start_step[3]}, h(n): {start_step[4]}\n")
    print("-" * 30)

    print(f"Solved in {len(solution_steps) - 1} moves:\n")
    for move, board, f, g, h in solution_steps[1:]:
        print(f"Move: {move}")
        print_board(board)
        print(f"Metrics -> f(n): {f}, g(n): {g}, h(n): {h}")
        print("-" * 30)
else:
    print("This puzzle state is unsolvable.")


Initial Board State:
2 8 3
1 6 4
7 0 5
Metrics -> f(n): 4, g(n): 0, h(n): 4

------------------------------
Solved in 5 moves:

Move: Up
2 8 3
1 0 4
7 6 5
Metrics -> f(n): 4, g(n): 1, h(n): 3
------------------------------
Move: Up
2 0 3
1 8 4
7 6 5
Metrics -> f(n): 5, g(n): 2, h(n): 3
------------------------------
Move: Left
0 2 3
1 8 4
7 6 5
Metrics -> f(n): 5, g(n): 3, h(n): 2
------------------------------
Move: Down
1 2 3
0 8 4
7 6 5
Metrics -> f(n): 5, g(n): 4, h(n): 1
------------------------------
Move: Right
1 2 3
8 0 4
7 6 5
Metrics -> f(n): 5, g(n): 5, h(n): 0
------------------------------


Implementing 4 Queens using Hill Climbing algorithm

In [15]:
import copy

class FourQueensProblem:
    def __init__(self, initial_state):
        self.initial_state = initial_state

    def get_cost(self, state):
        attacking_pairs = 0
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                    attacking_pairs += 1
        return attacking_pairs

    def get_neighbors(self, state):
        neighbors = []
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                neighbor_state = list(state)
                neighbor_state[i], neighbor_state[j] = neighbor_state[j], neighbor_state[i]
                cost = self.get_cost(neighbor_state)
                neighbors.append({
                    "state": neighbor_state,
                    "cost": cost
                })
        return neighbors

def format_state(state):
    return " ".join(map(str, state))

def hill_climbing(problem):
    current_state = problem.initial_state
    current_cost = problem.get_cost(current_state)
    first_step = True

    print(f"Initial State: {format_state(current_state)}\n")

    while True:
        # CRITICAL FIX: Stop immediately if the current state is a perfect solution (cost 0)
        if current_cost == 0:
            print(f"Goal Reached! Final State: {format_state(current_state)}")
            break

        if first_step:
            print(f"Solution: Neighbours of {format_state(current_state)} (cost is {current_cost}.)\n")
            first_step = False
        else:
            print(f"Neighbours of {format_state(current_state)} (cost is {current_cost}.)\n")

        neighbors = problem.get_neighbors(current_state)

        best_neighbor = None
        min_cost = current_cost

        for nb in neighbors:
            if nb["cost"] < min_cost:
                min_cost = nb["cost"]
                best_neighbor = nb
            elif nb["cost"] == min_cost and best_neighbor is None and nb["cost"] < current_cost:
                min_cost = nb["cost"]
                best_neighbor = nb

        for nb in neighbors:
            suffix = ""
            if best_neighbor and nb["state"] == best_neighbor["state"]:
                suffix = " Choose this neighbour."
            print(f"• {format_state(nb['state'])} cost is {nb['cost']}.{suffix}")

        print()

        # Stop if we are trapped in a local minimum/plateau
        if best_neighbor is None or best_neighbor["cost"] >= current_cost:
            print(f"Final State (Local Minimum): {format_state(current_state)}")
            break

        current_state = best_neighbor["state"]
        current_cost = best_neighbor["cost"]

if __name__ == "__main__":
    # Example input to try: 1 3 0 2 (Cost is 0 initially)
    # Another example: 2 0 3 1 (Will step towards cost 0)
    initial_input = [int(x) for x in input().strip().split()]
    print()

    problem = FourQueensProblem(initial_input)
    hill_climbing(problem)


3 1 2 0

Initial State: 3 1 2 0

Solution: Neighbours of 3 1 2 0 (cost is 2.)

• 1 3 2 0 cost is 1. Choose this neighbour.
• 2 1 3 0 cost is 1.
• 0 1 2 3 cost is 6.
• 3 2 1 0 cost is 6.
• 3 0 2 1 cost is 1.
• 3 1 0 2 cost is 1.

Neighbours of 1 3 2 0 (cost is 1.)

• 3 1 2 0 cost is 2.
• 2 3 1 0 cost is 2.
• 0 3 2 1 cost is 4.
• 1 2 3 0 cost is 4.
• 1 0 2 3 cost is 2.
• 1 3 0 2 cost is 0. Choose this neighbour.

Goal Reached! Final State: 1 3 0 2
